In [2]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

I0000 00:00:1788010831.264394   23505 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788010831.350671   23505 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788010833.850174   23505 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
with open("corpus.txt", "r") as f:
    text = f.read()
text

'I love machine learning\nI love deep learning\nMachine learning is powerful\nDeep learning is powerful\nArtificial intelligence is amazing\nMachine learning is amazing\nNeural networks are powerful\nNeural networks are useful\nDeep learning uses neural networks\nArtificial intelligence uses machine learning'

In [4]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
word_index = tokenizer.word_index
print(word_index)

{'learning': 1, 'machine': 2, 'is': 3, 'deep': 4, 'powerful': 5, 'neural': 6, 'networks': 7, 'i': 8, 'love': 9, 'artificial': 10, 'intelligence': 11, 'amazing': 12, 'are': 13, 'uses': 14, 'useful': 15}


In [5]:
sequences = tokenizer.texts_to_sequences([text])
print(sequences)

[[8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3, 12, 2, 1, 3, 12, 6, 7, 13, 5, 6, 7, 13, 15, 4, 1, 14, 6, 7, 10, 11, 14, 2, 1]]


In [6]:
input_sequences = []

for sequence in sequences:
    for i in range(1, len(sequence)):
        n_gram_sequence = sequence[:i+1]
        input_sequences.append(n_gram_sequence)

In [7]:
print(input_sequences)

[[8, 9], [8, 9, 2], [8, 9, 2, 1], [8, 9, 2, 1, 8], [8, 9, 2, 1, 8, 9], [8, 9, 2, 1, 8, 9, 4], [8, 9, 2, 1, 8, 9, 4, 1], [8, 9, 2, 1, 8, 9, 4, 1, 2], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3, 12], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3, 12, 2], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3, 12, 2, 1], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3, 12, 2, 1, 3], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5, 4, 1, 3, 5, 10, 11, 3, 12, 2, 1, 3, 12], [8, 9, 2, 1, 8, 9, 4, 1, 2, 1, 3, 5

In [8]:
max_sequence_len = max(
    len(sequence) for sequence in input_sequences
)
max_sequence_len

42

In [9]:
input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding="pre"
    )
)

In [10]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (41, 41)
y shape: (41,)


In [11]:
vocab_size = len(tokenizer.word_index) + 1

In [12]:
y = tf.keras.utils.to_categorical(
    y,
    num_classes=vocab_size
)

In [13]:
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        input_length=max_sequence_len - 1
    ),

    SimpleRNN(150),

    Dense(
        vocab_size,
        activation="softmax"
    )
])

/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1788010850.156389   23505 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [14]:
Dense(vocab_size, activation="softmax")

<Dense name=dense_1, built=False>

In [15]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X,
    y,
    epochs=200,
    verbose=1
)

Epoch 1/200


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.0976 - loss: 2.7705
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3902 - loss: 2.5200
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5122 - loss: 2.3730
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4878 - loss: 2.2688
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6341 - loss: 2.1416
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.7317 - loss: 2.0056
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.7317 - loss: 1.8823
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.7805 - loss: 1.7727
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8293 - loss: 1.6459
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.8293 - loss: 1.5393
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8537 - loss: 1.4316
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8537 - loss: 1.3381
E

In [18]:
token_list = tokenizer.texts_to_sequences(
    ["Deep learning uses "]
)[0]

In [19]:
token_list = pad_sequences(
    [token_list],
    maxlen=max_sequence_len - 1,
    padding="pre"
)

In [20]:
predicted = model.predict(token_list, verbose=0)

In [21]:
predicted_word_index = np.argmax(predicted, axis=1)[0]

In [22]:
reverse_word_index = {
    value: key
    for key, value in tokenizer.word_index.items()
}

In [23]:
predicted_word = reverse_word_index.get(
    predicted_word_index
)

In [24]:
print(predicted_word)

love


In [25]:
model.save("next_word_model.h5")